# 00-setup

_resource/00-setup.py — Delfos M1: Fundamentos, Arquitectura y Gobernanza
Invocado por cada modulo con: %run ./_resource/00-setup
Configura entorno, Unity Catalog y funciones compartidas. NO ejecutar directo.

### Widgets globales del workshop

In [0]:
# Celda 3
import re, json
from datetime import datetime
from pyspark.sql import functions as F, DataFrame
from pyspark.sql.window import Window
from typing import List

for _n, _d, _l in [
    ("s3_bucket", "nequi-workshop-tpt-202605", "S3 Bucket"),
    ("catalog",   "",           "Catalogo Unity Catalog (vacio = delfos_{nickname})"),
    ("dominio",   "pagos",      "Dominio de negocio"),
    ("nickname",  "tpt",        "Nickname / iniciales (sufijo de schemas, ej: jdoe)"),
]:
    try:
        dbutils.widgets.remove(_n)
    except:
        pass
    dbutils.widgets.text(_n, _d, _l)

try:
    dbutils.widgets.remove("reset")
except:
    pass
dbutils.widgets.dropdown("reset", "No", ["No", "Si — reiniciar datos"], "Reiniciar")

DOMINIO = dbutils.widgets.get("dominio").strip() or "pagos"
RESET   = dbutils.widgets.get("reset").startswith("Si")

_nick_raw = dbutils.widgets.get("nickname").strip().lower()
if not _nick_raw:
    _nick_raw = spark.sql("SELECT current_user()").collect()[0][0].split("@")[0]

NICK = re.sub(r"[^a-z0-9]", "", _nick_raw)[:15]
assert NICK, "No se pudo determinar el nickname"

_catalog_raw = dbutils.widgets.get("catalog").strip()
CATALOG = _catalog_raw if _catalog_raw else f"delfos_{NICK}"

for _wn, _wv, _wl in [
    ("catalog",  CATALOG, "Catalogo Unity Catalog (vacio = delfos_{nickname})"),
    ("nickname", NICK,    "Nickname / iniciales (sufijo de schemas, ej: jdoe)"),
]:
    try:
        dbutils.widgets.remove(_wn)
    except:
        pass
    dbutils.widgets.text(_wn, _wv, _wl)

### Credenciales y rutas S3

In [0]:
# Celda 5
_bw = dbutils.widgets.get("s3_bucket").strip()
if not _bw:
    _bw = "nequi-workshop-tpt-202605"

S3_BUCKET = _bw
_CREDS_OK = True

PATH_BRONZE = f"s3://{S3_BUCKET}/bronze/"
PATH_SILVER = f"s3://{S3_BUCKET}/silver/"
PATH_GOLD   = f"s3://{S3_BUCKET}/gold/"
PATH_CKPT   = f"s3://{S3_BUCKET}/_checkpoints/"
PATH_LOGS   = f"s3://{S3_BUCKET}/_audit_logs/"

### Dominios de Delfos en Unity Catalog

In [0]:
# Celda 7
DOMINIOS_DELFOS = {
    f"pagos_{NICK}"    : "Transacciones, pagos QR, transferencias — propietario: equipo-pagos",
    f"riesgo_{NICK}"   : "Modelos antifraude, alertas SARLAFT — propietario: equipo-riesgo",
    f"clientes_{NICK}" : "Perfil y segmentacion de usuarios — propietario: equipo-producto",
    f"canales_{NICK}"  : "App, corresponsal, QR, API — propietario: equipo-canales",
}
SCH_PAGOS    = f"pagos_{NICK}"
SCH_RIESGO   = f"riesgo_{NICK}"
SCH_CLIENTES = f"clientes_{NICK}"
SCH_CANALES  = f"canales_{NICK}"

if RESET:
    print(f"Limpiando catalogo '{CATALOG}'...")
    try:
        for _schema in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect():
            _s = _schema["databaseName"]
            if _s != "information_schema":
                spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{_s} CASCADE")
        spark.sql(f"DROP CATALOG IF EXISTS {CATALOG}")
    except Exception as _e:
        print(f"(catalogo no existia o ya estaba limpio: {_e})")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"USE CATALOG {CATALOG}")

def _setup_delfos(catalog: str) -> None:
    spark.sql(f"USE CATALOG {catalog}")
    for dominio, comentario in DOMINIOS_DELFOS.items():
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{dominio} COMMENT '{comentario}'")

    spark.sql(f'''
        CREATE TABLE IF NOT EXISTS {catalog}.{SCH_PAGOS}.transacciones (
            transaction_id  STRING    NOT NULL COMMENT 'UUID unico de la transaccion',
            user_id         STRING    NOT NULL COMMENT 'ID usuario (USR####)',
            monto           DOUBLE    COMMENT 'Monto en pesos colombianos COP',
            canal           STRING    COMMENT 'Canal: app | qr | corresponsal | api',
            ciudad          STRING    COMMENT 'Ciudad de origen',
            dispositivo     STRING    COMMENT 'ID dispositivo (DEV###)',
            ts              TIMESTAMP COMMENT 'Timestamp de la transaccion (ISO 8601)',
            capa            STRING    COMMENT 'Capa Medallion: bronze | silver | gold',
            es_fraude_real  BOOLEAN   COMMENT 'Label de verdad — solo para demo/validacion',
            _procesado_en   TIMESTAMP COMMENT 'Timestamp de ingesta por Delfos'
        ) USING DELTA
        PARTITIONED BY (capa)
        COMMENT 'Data Product: transacciones Nequi normalizadas. SLA 1h. SARLAFT.'
    ''')

_setup_delfos(CATALOG)
print(f"✅ Catalogo '{CATALOG}' listo")

### Funciones de utilidad compartidas (disponibles en todos los modulos)

In [0]:
# Calidad de datos
CANALES_VALIDOS = ["app", "qr", "corresponsal", "api"]

def aplicar_reglas_calidad(df: DataFrame) -> DataFrame:
    """Reglas de calidad Delfos: montos > 0, canal valido, ts tipado, sin duplicados."""
    return (df
        .filter(F.col("monto") > 0)
        .withColumn("ts", F.to_timestamp("ts"))
        .filter(F.col("canal").isin(CANALES_VALIDOS))
        .dropDuplicates(["transaction_id"])
        .withColumn("_procesado_en", F.current_timestamp())
    )

def zscore_por_usuario(df: DataFrame, umbral: float = 3.0) -> DataFrame:
    """Building block: z-score por usuario. Reutilizable en cualquier dominio."""
    v = Window.partitionBy("user_id")
    return (df
        .withColumn("_prom", F.avg("monto").over(v))
        .withColumn("_std",  F.stddev("monto").over(v))
        .withColumn("alerta_zscore",
            F.when(F.col("_std").isNotNull() & (F.col("_std") > 0),
                   (F.col("monto") - F.col("_prom")) / F.col("_std") > umbral
            ).otherwise(F.lit(False)))
    )

def frecuencia_ventana(df: DataFrame, ventana_seg: int = 600, max_tx: int = 5) -> DataFrame:
    """Building block: frecuencia en ventana deslizante. Reutilizable por cualquier equipo."""
    v = (Window.partitionBy("user_id")
               .orderBy(F.col("ts").cast("long"))
               .rangeBetween(-ventana_seg, 0))
    return (df
        .withColumn("_tx_ventana",      F.count("*").over(v))
        .withColumn("alerta_frecuencia", F.col("_tx_ventana") > max_tx)
    )

def modulo_header(num, titulo, subtitulo, icono, c1, c2, tiempo):
    sep = "=" * 70
    print(f"\n{sep}")
    print(f"  MODULO {num}  |  {tiempo}")
    print(f"  {titulo}")
    print(f"  {subtitulo}")
    print(f"{sep}\n")

def nota(html, tipo="info"):
    texto = re.sub(r"<[^>]+>", "", html).strip().replace("\n", " ")
    labels = {"info": "NOTA", "tip": "CONSEJO", "warn": "ATENCION"}
    print(f"\n[{labels.get(tipo, 'NOTA')}] {texto}\n")

def checkpoint(p, d, ref, url):
    def strip(s): return re.sub(r"<[^>]+>", "", s).strip()
    print("\n" + "-" * 70)
    print("REFLEXION")
    print(strip(p))
    print("\nDISCUSION")
    print(strip(d))
    print(f"\nReferencia: {ref}")
    print(f"URL: {url}")
    print("-" * 70 + "\n")

### Estado del entorno

In [0]:
_creds = "Cargadas desde Databricks Secrets" if _CREDS_OK else "No encontradas — verificar scope 'nequi'"

print("=" * 60)
print("  Delfos — Entorno listo")
print("=" * 60)
print(f"  Plataforma : Delfos (Nequi Data Platform)")
print(f"  Catalogo   : {CATALOG}")
print(f"  Nickname   : {NICK}")
print(f"  Schemas    : {' · '.join(DOMINIOS_DELFOS.keys())}")
print(f"  S3 Bucket  : {S3_BUCKET}")
print(f"  Cred. AWS  : {_creds}")
print(f"  Modulos    : 01 Data Mesh · 02 Arquitectura · 03 Building Blocks · 04 Priorizacion")
print("=" * 60)